# Pulse arrival times from the Crab pulsar
## Crab Pulsar Experiment Part 3.2

Use this notebook as a template for Part 3.2 of the Crab pulsar experiment.

In [6]:
# To begin, we import some libraries that we will need later.

# Some handy standard python libraries
import os

# The numpy library is very useful for many things
import numpy as np

# The interpolate library from scipy includes powerful interpolation routines
# including the Lagrange interpolation described in the lab script.
from scipy import interpolate

# Astropy provides many useful tools...
from astropy import coordinates as coord
from astropy import units as u
from astropy import constants as const
from astropy import time as astrotime

# The pyplot module from matplotlib will allow us to plot things.
from matplotlib import pyplot as plt
from math import pi

## Load Data
Here we load your ToA data as well as the file containing solar system barycentre coordinates.

In [7]:
# Connect (mount) your google drive as a virtual directory accessible by this python code.
from google.colab import drive         # Import the python module that allows you to access your google drive
rootpathdrive = '/content/drive'       # This will be the directory as which your google drive will be known
drive.mount(rootpathdrive)             # Now connect to this google drive.

# At first use it will ask you to click on a link, after which you should give permission
# for outside processes to access your google drive. A authorization code is generated which should be entered
#(this is explained in https://colab.research.google.com/notebooks/io.ipynb#scrollTo=XDg9OBaYqRMd).

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# A good test to do is if you can see the contents of the directory in which you work on your google drive.
# Here "My Drive" refers to the "root" of your google drive.
# By default your notebook should be in a directory called Colab Notebooks.
# This template assumes all files you want to read in are copied in the
# same directory. Note the slash at the end of the first line.

#==============================================================================================================
#pathcrabtemplate = rootpathdrive+'/'+'My Drive/Year 3 Lab/Pulsar/Crab_pulsar_template/' # Fin's Directory
pathcrabtemplate = rootpathdrive+'/'+'Shared with me/CrabPulsar/Crab_pulsar_template/' #Fin's 2
#pathcrabtemplate = rootpathdrive+'/'+'My Drive/Colab Notebooks/CrabPulsar/Crab_pulsar_template/' #Sara's directory
#==============================================================================================================

filelist = []
for (dirpath, dirnames, filenames) in os.walk(pathcrabtemplate):
    filelist.extend(filenames)
    break
print (filelist)   # Show the contents of your working directory. At least your notebook should show up here.

[]


In [9]:
# Specify the file with ToAs you want to work on.
toafile  = os.path.join(pathcrabtemplate,"20251111_054931_B0531+21.npz.toas.txt")
baryfile = os.path.join(pathcrabtemplate,"2025_ssb.txt") # ssb_2022 will work for all of 2022.

# Read the barycentre file... Read the numpy loadtxt page to understand what this does...
year, month, day, xpos, ypos, zpos = np.loadtxt(baryfile,unpack=True)
year = year
month = month
day = day
xpos = xpos
ypos = ypos
zpos = zpos

#print(year)
# @todo: Load in your ToAs in a similar way.
toa_day, toa_uncertainty = np.loadtxt(toafile,unpack=True)

FileNotFoundError: /content/drive/Shared with me/CrabPulsar/Crab_pulsar_template/2025_ssb.txt not found.

In [ ]:
#print(toa_day)
#print(year, month, day)

In [ ]:
from astropy.time import Time

def ymd_to_mjd_array_astropy(year, month, day):
    """
    Convert year, month, day (0h UT) to Modified Julian Date (MJD) using Astropy.
    Supports scalar or NumPy array inputs.
    """
    # Create datetime strings in ISO format
    dates = [f"{int(y)}-{int(m):02d}-{int(d):02d}" for y, m, d in zip(year, month, day)]

    # Create Astropy Time object
    t = Time(dates, format='iso')

    # Return MJD
    return t.mjd

# Use Astropy for conversion
barycentre_mjd = ymd_to_mjd_array_astropy(year, month, day)

# Step 2: Apply the date window
lower_limit = 60968
upper_limit = 60971

measurement_mask = (barycentre_mjd >= lower_limit) & (barycentre_mjd <= upper_limit)

# Step 3: Filter all arrays
barycentre_mjd_new = barycentre_mjd[measurement_mask]
xpos_new = xpos[measurement_mask]
ypos_new = ypos[measurement_mask]
zpos_new = zpos[measurement_mask]

print(barycentre_mjd_new)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True)

# X position
axes[0].scatter(barycentre_mjd_new, xpos_new, s=10)
axes[0].set_ylabel("X Position")
axes[0].set_title("Position vs Time")

# Y position
axes[1].scatter(barycentre_mjd_new, ypos_new, s=10)
axes[1].set_ylabel("Y Position")

# Z position
axes[2].scatter(barycentre_mjd_new, zpos_new, s=10)
axes[2].set_ylabel("Z Position")
axes[2].set_xlabel("Time (Modified Julian Date)")

plt.tight_layout()
plt.show()


## Interpolation
The Earth-barycentre vector is only given once per day (at 0 UT) in your input file. Therefore you will need to [interpolate](https://en.wikipedia.org/wiki/Interpolation) to get the vector at the time of each of your ToAs.

The below is an example code showing interpolation of a simple sinusoid function. Make sure you understand what this code is doing, then replace it with a function to interpolate your x, y and z positions at the times of your ToAs.

In [ ]:
# Interpolation Valyes
start_point = 0
interp_number=3

# For x coordinate
x = barycentre_mjd_new
y_x = xpos_new

interp_function_x = interpolate.lagrange(x[start_point:start_point+interp_number],\
                                        y_x[start_point:start_point+interp_number])

x2 = np.linspace(x[start_point], x[start_point + interp_number - 1], 100)
y2_x = interp_function_x(x2)

plt.plot(x, y_x, 'o', color='green')
plt.plot(x2, y2_x, '.', color='blue')
plt.ylim(-1.1, 1.1)
plt.title("Interpolation - X")
plt.show()

xpos_at_times = interp_function_x(toa_day)

# For y coordinate
y_y = ypos_new

interp_function_y = interpolate.lagrange(x[start_point:start_point+interp_number],\
                                        y_y[start_point:start_point+interp_number])

y2_y = interp_function_y(x2)

plt.plot(x, y_y, 'o', color='green')
plt.plot(x2, y2_y, '.', color='red')
plt.ylim(-1.1, 1.1)
plt.title("Interpolation - Y")
plt.show()

ypos_at_times = interp_function_y(toa_day)

# For z coordinate
y_z = zpos_new

interp_function_z = interpolate.lagrange(x[start_point:start_point+interp_number],\
                                        y_z[start_point:start_point+interp_number])

y2_z = interp_function_z(x2)

plt.plot(x, y_z, 'o', color='green')
plt.plot(x2, y2_z, '.', color='purple')
plt.ylim(-1.1, 1.1)
plt.title("Interpolation - Z")
plt.show()

zpos_at_times = interp_function_z(toa_day)

# Print or save all interpolated coordinates
print("Interpolated coordinates at TOA days:")
print(f"X positions: {xpos_at_times}")
print(f"Y positions: {ypos_at_times}")
print(f"Z positions: {zpos_at_times}")


coordinates_array = np.column_stack((xpos_at_times, ypos_at_times, zpos_at_times))

In [ ]:
# Example: check how well the interpolation predicts nearby actual data points
actual_values = ypos_new[start_point:start_point+interp_number+1]  # one more real point
predicted_values = interp_function_y(x[start_point:start_point+interp_number+1])
residuals = actual_values - predicted_values
error = np.sqrt(np.mean(residuals**2))  # RMS error
print("RMS interpolation error:", error)


## Compute the Earth delay

This is the delay due to the fact that the observatory is not at the centre of the Earth. For this, you need to know the elevation angle above the horizon (see the lab script for more details).

The [Astropy](http://www.astropy.org/) library provides very powerful tools to do things like coordinate transforms. Here we want to convert from equitorial (RA and Dec) coordinate system to an observatory-based [AltAz](http://docs.astropy.org/en/stable/api/astropy.coordinates.AltAz.html) coordinate system. Make sure you are familiar with these two coordinate systems. The conversion from one to the other requries knowing the location of your telecsope and also the time the observations were taken.

The following libraries have been imported for your convinience:
 * ``astropy.coordinate`` has been imported as ``coord``
 * ``astropy.time`` has been imported as ``astrotime``


In [ ]:
from astropy import constants as const


# Crab Pulsar position
pulsarpos = coord.SkyCoord(ra="05:34:31.9", dec="+22:00:52.0",
                           unit=(u.hourangle, u.deg))

lovellpos = coord.EarthLocation(
    lat="53:14:09", lon="-2:18:23", height=126.3*u.m
)

times = Time(toa_day, format='mjd', scale='utc')

# Transform to AltAz
altaz = pulsarpos.transform_to(coord.AltAz(obstime=times, location=lovellpos))

# Check output
print("Altitudes (deg):", altaz.alt)
print("Azimuths (deg):", altaz.az)

# Optional plot
plt.plot(times.mjd, altaz.alt.deg, 'o-')
plt.xlabel("MJD")
plt.ylabel("Altitude (degrees)")
plt.title("Crab Pulsar Elevation at Lovell Telescope")
plt.grid(True)
plt.show()


# Get altitude in radians
elevation_rad = altaz.alt.to(u.rad).value  # numeric array in radians

# Constants
R_earth = const.R_earth.value  # meters
c = const.c.value              # m/s

# Compute Earth delay
earth_delay = (R_earth * np.sin(elevation_rad)) / c  # seconds

print("Earth delay (s):", earth_delay)


## Compute the Roemer delay

This is the delay corresponding to the distance of the centre of the Earth to the barycentre.
$$
\Delta_{Rsun} = \frac{1}{c} \vec{r}_{vec} \cdot \hat{\vec{s}} = -\frac{1}{c} \left( \vec{r}_{SSB} + \vec{r}_{EO} \right) \cdot \hat{\vec{s}}
$$


In [ ]:
c = const.c

# Barycentre positions at ToA and subsequently our observation coords (centre of earth)
psr_x = xpos_at_times
psr_y = ypos_at_times
psr_z = zpos_at_times
r_ssb = np.column_stack((xpos_at_times, ypos_at_times, zpos_at_times))  # Earth to barycentre vector at TOA

# Converting pulsar's RA and Dec coords into cartesian
pulsar_cart = pulsarpos.icrs.cartesian
s_hat = np.array([pulsar_cart.x.value,
                  pulsar_cart.y.value,
                  pulsar_cart.z.value])
s_hat /= np.linalg.norm(s_hat)  # ensure it's a unit vector

r_ssb_q = r_ssb * const.au  # corrected r_ssb in metres

# Fix: Add units to r_sine to make it consistent
r_sine = (R_earth * np.sin(elevation_rad)) * u.m  # Add units explicitly

# Calculate the dot product and ensure proper units
dot_product = np.dot(r_ssb_q, s_hat)  # This has units of meters

roemer_delay = (1/c) * (dot_product + r_sine)

print(roemer_delay)
print("Roemer delay (s) for each TOA:")
print(roemer_delay)

plt.plot(toa_day, roemer_delay.to(u.ms), 'o-')
plt.xlabel("MJD")
plt.ylabel("Roemer Delay (ms)")
plt.title("Roemer Delay for Crab Pulsar")
plt.grid(True)
plt.show()



In [ ]:
# Convert Time object to MJD values (plain numbers)
times_mjd = times.mjd  # This gives you the numerical MJD values

print("Times MJD values:", times_mjd)

# Convert Roemer delay from seconds to days (since MJD is in days)
roemer_delay_days = roemer_delay.to(u.day)

# Now add them properly
corrected_time_mjd = times_mjd + roemer_delay_days.value

print("Corrected times (MJD):", corrected_time_mjd)

# If you want to convert back to Time object:
corrected_time = Time(corrected_time_mjd, format='mjd')
print("Corrected Time objects:", corrected_time)

## Residuals from a fixed period model - WORKING PROGRESS



In [ ]:
# # Reference (The first corrected TOA)
# t_0_mjd = corrected_time_mjd[0]

# # Calculate time difference (t - t0) in MJD days
# delta_t_days = corrected_time_mjd - t_0_mjd

# # Convert time difference to seconds
# delta_t_seconds = delta_t_days * SECONDS_PER_DAY
# time_elapsed = delta_t_seconds

# #print("Time elapsed (s):", time_elapsed)

# # Calculate the EXPECTED number of rotations (N) based on the model
# # N = (t - t0) / P_model
# N = delta_t_seconds / period_guess
# #print("N:", N)
# # Calculate the fractional part of N (the residual in PHASE)
# # fractional_part = N - floor(N)
# N_integer = np.round(N)
# #print("N_integer:", N_integer)


# N_fraction = N - N_integer

# # Calculate the RESIDUAL in TIME (seconds)
# # Residual (dt) = fractional_part * P_model
# residuals = N_fraction #* period_guess

# print("Length of residuals (old)", len(residuals))
# print("Values in residuals (old)", residuals)

# print("Length of time elapsed:", len(time_elapsed))
# print("Time elapsed:", time_elapsed)

# residuals_new = np.concatenate([
#     residuals[0:5],
#     residuals[5:15] + 1,
#     residuals[15:28] + 2,
#     residuals[28:32] + 3
# ])


# # # Linear fitting

# # # Perform a linear fit (polynomial of degree 1) to the residuals vs. time
# # # polyfit returns [m, c], where m is the slope and c is the y-intercept.
# # # We fit Residuals (Y) vs. Time Elapsed (X)
# (slope_m, intercept_c) = np.polyfit(time_elapsed, residuals_new, 1)

# fit_line = slope_m * time_elapsed + intercept_c

# plt.figure(figsize=(10, 6))
# plt.plot(time_elapsed, residuals_new, 'o', label='Calculated Residuals')
# plt.plot(time_elapsed, fit_line, 'r-', label=f'Linear Fit (Slope = {slope_m:.2e})')
# plt.axhline(0, color='k', linestyle='--', linewidth=0.8)

# plt.xlabel('Time Elapsed from $t_0$ (seconds)')
# plt.ylabel('Residuals (No Units?)')
# plt.title(f'Crab Pulsar Timing Residuals')
# plt.legend()
# plt.grid(True)
# plt.show()

# print("Gradient: ", slope_m)



Since $$m = f_{test} - f_{true}$$
$$ P_{true} = \frac{-1}{m - \frac{1}{P_{test}}}
$$

and when we plot $P_{true}$ against the elapsed time we can see that it worked

In [ ]:
# P_true = -1/(slope_m - 1/period_guess)
# print("P_true:", P_true)

# # Reference (The first corrected TOA)
# t_0_mjd = corrected_time_mjd[0]

# # Calculate time difference (t - t0) in MJD days
# delta_t_days = corrected_time_mjd - t_0_mjd

# # Convert time difference to seconds
# delta_t_seconds = delta_t_days * SECONDS_PER_DAY
# time_elapsed = delta_t_seconds

# #print("Time elapsed (s):", time_elapsed)

# # Calculate the EXPECTED number of rotations (N) based on the model
# # N = (t - t0) / P_model
# N = delta_t_seconds / P_true
# #print("N:", N)
# # Calculate the fractional part of N (the residual in PHASE)
# # fractional_part = N - floor(N)
# N_integer = np.round(N)
# #print("N_integer:", N_integer)


# N_fraction = N - N_integer

# # Calculate the RESIDUAL in TIME (seconds)
# # Residual (dt) = fractional_part * P_model
# residuals = N_fraction

# #fit_line = slope_m * time_elapsed + intercept_c

# plt.figure(figsize=(10, 6))
# plt.plot(time_elapsed, residuals, 'o', label='Calculated Residuals')
# plt.axhline(0, color='k', linestyle='--', linewidth=0.8)

# plt.xlabel('Time Elapsed from $t_0$ (seconds)')
# plt.ylabel('Residuals (Number of Spins)')
# plt.title(f'Crab Pulsar Timing Residuals (testing improved value)')
# plt.legend()
# plt.grid(True)
# plt.show()

# #print("Gradient: ", slope_m)

In [ ]:
# Constants
SECONDS_PER_DAY = 86400.0

# Initial model period estimate (in seconds)

print(corrected_time_mjd)

period_guess = 0.033847826  # seconds

indices_to_remove = [0, 2, 3, 4, 13, 15, 17, 18, 21, 32, 41]
corrected_time_mjd = np.delete(corrected_time_mjd, indices_to_remove)
#time_elapsed = np.delete(time_elapsed, indices_to_remove)

print(corrected_time_mjd)

#Time elapsed calculations

t0_mjd = corrected_time_mjd[0]
delta_t_seconds = (corrected_time_mjd - t0_mjd) * SECONDS_PER_DAY
time_elapsed = delta_t_seconds

# Initial residuals using period_guess
N = time_elapsed / period_guess
N_int = np.round(N)                    # best-fit integer cycle count
residuals = N - N_int                  # phase residuals in cycles

print("Length of residuals:", len(residuals))
print("Residuals:", residuals)
print("Length of time_elapsed:", len(time_elapsed))


#Cleaning data (manual)

residuals_new = np.concatenate([
    residuals[0:5],            # 5 points
    residuals[5:15] + 1,       # 10 points
    residuals[15:28] + 2,      # 13 points
    residuals[28:32] + 3       # 4 points
])


#Linear fit and plotting
slope_m, intercept_c = np.polyfit(time_elapsed, residuals_new, 1)
fit_line = slope_m * time_elapsed + intercept_c

plt.figure(figsize=(10, 6))
plt.plot(time_elapsed, residuals_new, 'o', label='Unwrapped Residuals')
plt.plot(time_elapsed, fit_line, 'r-', label=f'Fit (slope = {slope_m:.3e})')
plt.axhline(0, color='k', linestyle='--')

plt.xlabel('Time Elapsed from $t_0$ (s)')
plt.ylabel('Residuals (cycles)')
plt.title('Crab Pulsar Timing Residuals (Unwrapped)')
plt.legend()
plt.grid(True)
plt.show()

print("Slope m =", slope_m)

# Applying corrected period

P_true = -1 / (slope_m - 1/period_guess)
print("Improved Period P_true =", P_true)

N2 = time_elapsed / P_true
N2_int = np.round(N2)
residuals_corrected = N2 - N2_int

plt.figure(figsize=(10, 6))
plt.plot(time_elapsed, residuals_corrected, 'o', label='Corrected Residuals')
plt.axhline(0, color='k', linestyle='--')

plt.xlabel('Time Elapsed from $t_0$ (s)')
plt.ylabel('Residuals (cycles)')
plt.title('Residuals Using Improved Period')
plt.legend()
plt.grid(True)
plt.show()
